# Agricultural Super-Resolution GAN - RunPod

**First-time setup:** Open a terminal in JupyterLab and run:
```bash
cd /workspace
git clone https://github.com/VikramK2010/SuperResolution super-gan
cd super-gan
bash setup.sh
```
This installs dependencies and downloads the PlantVillage dataset. You only need to do this once.

## 1. Verify Setup

In [ ]:
import os
import torch

# Make sure we're in the right directory
os.chdir("/workspace/super-gan")

print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

# Verify the dataset is ready
train_dir = "data/plantvillage/color_images/train"
assert os.path.isdir(train_dir), (
    f"Dataset not found at {train_dir}. "
    "Run 'bash setup.sh' in the terminal first."
)
num_classes = len(os.listdir(train_dir))
print(f"Dataset OK: {num_classes} classes found")

## 2. Load and Explore Data

In [ ]:
from training_utils import (
    load_class_names,
    load_plant_balanced_image_file_sample,
    build_plant_dict,
    data_summary,
    summarize_image_attributes,
)

train_dir = "data/plantvillage/color_images/train"

class_names = load_class_names(train_dir)
plant_dict = build_plant_dict(class_names)
sample_image_files, sample_labels, sample_classes = load_plant_balanced_image_file_sample(
    train_dir, plant_dict, 500
)

In [ ]:
data_summary(sample_image_files, sample_labels, class_names)
summarize_image_attributes(sample_image_files)

## 3. Create Dataset and DataLoader

In [ ]:
import dataset_utils
from torch.utils.data import DataLoader

dataset = dataset_utils.SuperResolutionDataset(image_files=sample_image_files)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True, num_workers=4, pin_memory=True)

print(f"Dataset: {len(dataset)} images")
print(f"Batches per epoch: {len(dataloader)}")

## 4. Import Models

In [ ]:
from generators import SimpleGenerator, ResidualGenerator, AttentionMedicalGenerator
from agricultural_sr import AgriculturalAttentionGenerator, AgriculturalDiscriminator, AgriculturalTrainer
from training_classes import AdaptiveTrainer, MedicalTrainer, EnhancedTrainer
from discriminators import SimpleDiscriminator, SpectralNormDiscriminator
import training_loop

print("All models imported successfully.")

## 5. Weights & Biases (Optional)

Run the cell below to enable experiment tracking with [wandb.ai](https://wandb.ai).
It will print a login link — click it, copy your API key, and paste it back here.

If you don't want to use W&B, skip this cell. Training will still work fine without it.

In [ ]:
import wandb

# This will prompt you for an API key the first time.
# 1. Go to https://wandb.ai/authorize (create a free account if you don't have one)
# 2. Copy the API key
# 3. Paste it below when prompted
wandb.login()

USE_WANDB = True
WANDB_PROJECT = "Agricultural-Super-Resolution"  # Change this to whatever you want

print(f"W&B enabled. View your runs at: https://wandb.ai/{WANDB_PROJECT}")

## 6. Train

In [ ]:
# If you skipped the W&B cell above, these defaults keep training working without it
if "USE_WANDB" not in dir():
    USE_WANDB = False
    WANDB_PROJECT = None

generator, discriminator = training_loop.enhanced_training_loop(
    dataset,
    dataloader,
    AgriculturalTrainer,
    AgriculturalAttentionGenerator,
    AgriculturalDiscriminator,
    model_name="ag_gen_and_dis_model_500_per_class_sample_pv",
    num_epochs_per_block=2,
    total_epochs=10,
    checkpoint_dir="checkpoints",
    use_wandb=USE_WANDB,
    wandb_project_name=WANDB_PROJECT,
)

## 7. Save Final Models

In [ ]:
import torch

os.makedirs("checkpoints", exist_ok=True)
torch.save(generator.state_dict(), "checkpoints/ag_generator.pth")
torch.save(discriminator.state_dict(), "checkpoints/ag_discriminator.pth")
print("Models saved to checkpoints/")

## 8. Experimenting with Other Model Configs

Below are ready-to-run cells for different model combinations. Each one is independent —
just run whichever you want to try. Results are saved to separate checkpoint folders so
they won't overwrite each other.

**Available generators** (all RGB, all accept `num_residual_blocks`):
- `ResidualGenerator` — Simpler baseline, no attention. Faster to train.
- `AgriculturalAttentionGenerator` — Attention mechanisms tuned for crop images. (used above)

**Available discriminators:**
- `SimpleDiscriminator` — Basic 4-layer CNN.
- `SpectralNormDiscriminator` — Spectral normalization + self-attention. More stable training.
- `AgriculturalDiscriminator` — Spectral norm tuned for crop images. (used above)

**Available trainers** (use with `enhanced_training_loop`):
- `EnhancedTrainer` — General-purpose with pixel + perceptual + combined losses.
- `AgriculturalTrainer` — Adds crop-specific losses: color preservation, texture, SSIM. (used above)

**Key parameter: `residual_blocks_list`** — pass a list like `[4, 8, 16]` to train
multiple models with increasing capacity in a single run.

### 8a. Baseline: ResidualGenerator + EnhancedTrainer

A simpler model with no attention layers. Good as a comparison point to see how much
the agricultural-specific architecture and losses actually help.

In [ ]:
gen_baseline, disc_baseline = training_loop.enhanced_training_loop(
    dataset,
    dataloader,
    EnhancedTrainer,
    ResidualGenerator,
    SpectralNormDiscriminator,
    model_name="baseline_residual",
    num_epochs_per_block=2,
    total_epochs=10,
    checkpoint_dir="checkpoints",
    use_wandb=USE_WANDB,
    wandb_project_name=WANDB_PROJECT,
)

### 8b. Agricultural Attention with More Residual Blocks

Trains three models with 4, 8, and 16 residual blocks in one run.
More blocks = more capacity, but slower training and more GPU memory.
The training loop will save separate checkpoints for each.

In [ ]:
gen_multi, disc_multi = training_loop.enhanced_training_loop(
    dataset,
    dataloader,
    AgriculturalTrainer,
    AgriculturalAttentionGenerator,
    AgriculturalDiscriminator,
    residual_blocks_list=[4, 8, 16],
    model_name="ag_attention_multiblock",
    num_epochs_per_block=2,
    total_epochs=10,
    checkpoint_dir="checkpoints",
    use_wandb=USE_WANDB,
    wandb_project_name=WANDB_PROJECT,
)

### 8c. Mix and Match: ResidualGenerator + AgriculturalTrainer

Uses the simpler generator but keeps the agricultural-specific loss functions
(color preservation, texture, SSIM). A good way to test whether the custom losses
help even without the attention-based architecture.

In [ ]:
gen_mix, disc_mix = training_loop.enhanced_training_loop(
    dataset,
    dataloader,
    AgriculturalTrainer,
    ResidualGenerator,
    AgriculturalDiscriminator,
    model_name="residual_ag_losses",
    num_epochs_per_block=2,
    total_epochs=10,
    checkpoint_dir="checkpoints",
    use_wandb=USE_WANDB,
    wandb_project_name=WANDB_PROJECT,
)